# Suite P3CHK — Checking Python 3 modules against versions, and as trees

Which versions of Python 3 have which constructs, as the port records it: `Python3.Standards` records the versions
`Programs.Python3` covers as data, each holding the statement and clause kinds it has, as mbse-expressions' dialects
record the terms' (mbse-basis CAP-268), and its statements' grammar clauses, which each version allows or forbids, as
mbse-expressions' dialects record their terms' (mbse-basis, grammar clauses by version);
`Python3.Check.check(store, path, version)` lists the constructs a stored module uses that a version lacks, each
located by where it stands in the module's tree (`body[0].value`, `body[1].decorator_list[0]`), reading those
standards and grammar clauses as data; `Python3.Check.validate(store, path)` lists a
module's problems as a tree, each kind's own rules and its terms' as their dialects validate them. Where a statement or
a clause may stand is no rule of `validate`'s: the schemas refuse a misplaced one on write. Ported from suite PY
(availability by version, 3.0 to 3.15, and each kind's own checks) and PYPRT-06 (printing checked the version).

A module's tree is one value, its `body`, each statement a value object. Each case parses source with
`Python3.Parse.parse` into one store, `Python3.store()` prepared by `Standards.prepare`, or, where the parser would not
write the tree, writes the module's `body` by hand in mbse-schemas' builder syntax, as `Parse` writes it: a list of
branches of `Programs.Python3.Statement`, each tagged by its kind in lower case (`expr`, `functiondef`), one whose tag is
a Python keyword reached with `getattr(u, "if")`; a `try`'s `handlers` branches of `Programs.Python3.Handler`
(`excepthandler`, `error`), a `match`'s `cases` of `Programs.Python3.Case` (`matchcase`, `error`).

In [ ]:
import itertools
from pathlib import Path

from functools import reduce

from mbse.Expressions.Framework import Objects, Standards as ExpressionStandards
from mbse.Expressions.Python.Python3 import Expressions as P
from mbse.Programs import Python3
from mbse.Programs.Python3 import Check, Parse, Print, Standards
from mbse.Programs.Python3.Statements import KINDS, LANGUAGE
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Standards.prepare(Python3.store())
PATHS = (f"case{n}.py" for n in itertools.count())


def parsed(text):
    """The path of a new module holding `text`, parsed into the store."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return path


def check(text, version):
    return Check.check(STORE, parsed(text), version)


def needs(text, minor, *problems):
    """`text` is fine in Python 3.`minor`, and has `problems` in the version before."""
    path = parsed(text)
    assert Check.check(STORE, path, (3, minor)) == [], Check.check(STORE, path, (3, minor))
    assert Check.check(STORE, path, (3, minor - 1)) == list(problems), Check.check(STORE, path, (3, minor - 1))


def written(*body):
    """The path of a new module whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.Python3.Statement`, as `Parse.parse` writes one."""
    path = next(PATHS)

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "module": txn.Programs.Python3.Module})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.module)}),
                lambda _: _.module.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return path


def validate(*body):
    return Check.validate(STORE, written(*body))


# Statements that blocks below hold
PASS = lambda u: getattr(u, "pass")(lambda s: s)
COMMENT = lambda u: u.comment(lambda s: s.text(" c"))

## P3CHK-01 · The versions as data: `Programs.Python3` covers Python 3.0 to 3.15, each holding the statement kinds it has, beside mbse-expressions' dialects' own; run again, nothing changes

In [ ]:
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    standards = ExpressionStandards.standards(language)
    assert list(standards) == [f"Python 3.{minor}" for minor in range(16)]
    assert ExpressionStandards.find(language, "Python 3.12") is standards["Python 3.12"]
    assert ExpressionStandards.properties(standards["Python 3.12"]) == {"bodies": ("org.python",),
                                                                         "revision": ("version", "3.12")}
    assert ExpressionStandards.designation_of(standards["Python 3.10"]) == "Python 3.10"

    def kinds(minor):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(standards[f"Python 3.{minor}"])}

    assert kinds(15) == set(KINDS) and len(KINDS) == 31
    assert set(KINDS) - kinds(0) == {"AsyncFor", "AsyncWith", "AsyncFunctionDef", "AnnAssign", "Match", "MatchCase",
                                     "TryStar", "TypeAlias"}
    assert kinds(5) - kinds(4) == {"AsyncFor", "AsyncWith", "AsyncFunctionDef"} and kinds(6) - kinds(5) == {"AnnAssign"}
    assert kinds(10) - kinds(9) == {"Match", "MatchCase"} and kinds(11) - kinds(10) == {"TryStar"}
    assert kinds(12) - kinds(11) == {"TypeAlias"} and kinds(15) == kinds(12)
    assert sorted(k for k, versions in Standards.ONLY.items() if versions == Standards.VERSIONS[5:]) == [
        "AsyncFor", "AsyncFunctionDef", "AsyncWith"] and Standards.ONLY["TypeAlias"] == Standards.VERSIONS[12:]
    assert not hasattr(Standards, "SINCE")  # membership, no minimum version
    # The terms' versions are mbse-expressions' own, each dialect's by the same designations
    for dialect in Standards.DIALECTS[1:]:
        assert set(ExpressionStandards.standards(STORE.singleton(dialect))) <= set(standards), dialect
    patterns = ExpressionStandards.standards(STORE.singleton("Expressions.Python.Patterns"))
    assert list(patterns)[0] == "Python 3.10"  # patterns arrived in 3.10
    expressions = ExpressionStandards.standards(STORE.singleton("Expressions.Python.Python3"))
    tags = {kind.name for kind in ExpressionStandards.kinds(expressions["Python 3.7"])}
    assert "Expressions.Python.Python3.JoinedStr" in tags and "Expressions.Python.Python3.NamedExpr" not in tags
# The language is one of the framework's dialects, which is what may own a standard
dialects = STORE.registered("Expressions.Framework.Dialect")
assert any(b.name == "programs_python3" and b.schema.name == "Programs.Python3.Language" for b in dialects.branches)
# Run again, each writes nothing: the same versions, each with the same kinds
Standards.prepare(STORE)
with STORE.transaction():
    again = ExpressionStandards.standards(STORE.singleton(LANGUAGE))
    assert list(again) == list(standards) and len(ExpressionStandards.kinds(again["Python 3.15"])) == 31
assert Standards.designation(14) == "Python 3.14" and list(Standards.MINORS) == list(range(16))
print("ok")

## P3CHK-02 · Where kinds and grammar clauses exist: which versions of Python 3 have each, each located by where it stands in the module's tree

In [ ]:
needs("def g():\n    yield from x\n", 3, "body[0].body[0].value: yieldfrom needs Python 3.3")
needs("await a\n", 5, "body[0].value: await needs Python 3.5")
needs("a @ b\n", 5, "body[0].value: binop.op '@' needs Python 3.5")
needs("a @= b\n", 5, "body[0].op: AugAssign.op '@' needs Python 3.5")
needs("async def f():\n    async with a:\n        async for x in y:\n            pass\n", 5,
      "body[0]: AsyncFunctionDef needs Python 3.5", "body[0].body[0]: AsyncWith needs Python 3.5",
      "body[0].body[0].body[0]: AsyncFor needs Python 3.5")
needs("a: int\n", 6, "body[0]: AnnAssign needs Python 3.6")
needs("f'x'\n", 6, "body[0].value: joinedstr needs Python 3.6")
needs("[x async for x in y]\n", 6, "body[0].value: comprehension.is_async needs Python 3.6")
needs("(a := b)\n", 8, "body[0].value: namedexpr needs Python 3.8")
needs("def f(a, /):\n    pass\n", 8, "body[0].args: arguments.posonlyargs needs Python 3.8")
needs("lambda a, /: a\n", 8, "body[0].value: arguments.posonlyargs needs Python 3.8")
needs("f'{a=}'\n", 8, "body[0].value: formattedvalue.debug needs Python 3.8")
needs("@d[0]\ndef f():\n    pass\n", 9,
      "body[0].decorator_list[0]: FunctionDef with a decorator holding any term needs Python 3.9")
needs("@d\n@(d)\nasync def f():\n    pass\n", 9,
      "body[0].decorator_list[1]: AsyncFunctionDef with a decorator holding any term needs Python 3.9")
assert check("@d\n@d.e(1)\nclass C:\n    pass\n", (3, 8)) == []
needs("match a:\n    case None:\n        pass\n", 10, "body[0]: Match needs Python 3.10",
      "body[0].cases[0]: MatchCase needs Python 3.10", "body[0].cases[0].pattern: matchsingleton needs Python 3.10")
needs("try:\n    pass\nexcept* E:\n    pass\n", 11, "body[0]: TryStar needs Python 3.11")
needs("a[*b]\n", 11, "body[0].value: subscript with an unpacked index needs Python 3.11")
needs("x = a[i, *b]\n", 11, "body[0].value: subscript with an unpacked index needs Python 3.11")
assert check("a[i,]\na[i]\n", (3, 10)) == []
needs("type X = int\n", 12, "body[0]: TypeAlias needs Python 3.12")
needs("class C[T]:\n    pass\n", 12, "body[0].type_params: ClassDef.type_params needs Python 3.12",
      "body[0].type_params[0]: typevar needs Python 3.12")
needs("def f[T](x: T) -> T:\n    pass\n", 12, "body[0].type_params: FunctionDef.type_params needs Python 3.12",
      "body[0].type_params[0]: typevar needs Python 3.12")
needs("type X[**P = a] = int\n", 13, "body[0].type_params[0]: paramspec.default_value needs Python 3.13")
needs("type X[*Ts = a] = int\n", 13, "body[0].type_params[0]: typevartuple.default_value needs Python 3.13")
needs("type X[T: int = str] = T\n", 13, "body[0].type_params[0]: typevar.default_value needs Python 3.13")
needs("t'x'\n", 14, "body[0].value: templatestr needs Python 3.14")
# Not found: an except clause's types written bare, which the tree does not record (Parse writes a tuple either way)
assert check("try:\n    pass\nexcept A, B:\n    pass\n", (3, 13)) == []
needs("lazy import m\n", 15, "body[0].is_lazy: Import.is_lazy needs Python 3.15")
needs("lazy from . import a\n", 15, "body[0].is_lazy: ImportFrom.is_lazy needs Python 3.15")
needs("[*x for x in y]\n", 15, "body[0].value: listcomp of unpacked elements needs Python 3.15")
needs("{*x for x in y}\n", 15, "body[0].value: setcomp of unpacked elements needs Python 3.15")
needs("(*x for x in y)\n", 15, "body[0].value: generatorexp of unpacked elements needs Python 3.15")
needs("{**x for x in y}\n", 15, "body[0].value: dictcomp of unpacked mappings needs Python 3.15")
assert check("{x: y for x in z}\n[x for x in y]\n", (3, 14)) == []
# A statement's problems, then its properties', then its children's; each once, in source order
assert check("x = t'a'\nif (y := t'b'):\n    lazy import m\n", (3, 7)) == [
    "body[0].value: templatestr needs Python 3.14", "body[1].test: namedexpr needs Python 3.8",
    "body[1].test: templatestr needs Python 3.14", "body[1].body[0].is_lazy: Import.is_lazy needs Python 3.15"]
assert check("(a := (b := c))\n", (3, 7)) == ["body[0].value: namedexpr needs Python 3.8"]
assert check("class C(B, metaclass=M[*t]):\n    x: f'{a!r}'\n", (3, 5)) == [
    "body[0].keywords[0]: subscript with an unpacked index needs Python 3.11", "body[0].body[0]: AnnAssign needs Python 3.6",
    "body[0].body[0].annotation: joinedstr needs Python 3.6"]
print("ok")

## P3CHK-03 · The version by `(3, minor)` or by name; versions the language does not cover, a store not prepared, and a path the workspace does not hold

In [ ]:
from mbse.Programs.Framework import Model
path = parsed("type X = int\n")
assert Check.check(STORE, path, (3, 11)) == Check.check(STORE, path, "Python 3.11") == [
    "body[0]: TypeAlias needs Python 3.12"]
assert Check.check(STORE, path, (3, 12)) == Check.check(STORE, path, "Python 3.12") == []
for version in ((3, 16), (2, 7), (4, 0), "Python 4.0", "3.12"):
    with raises(LookupError, match="Programs.Python3 covers no version"):
        Check.check(STORE, path, version)
with raises(LookupError):
    Check.check(STORE, "nowhere.py", (3, 12))
with raises(LookupError):
    Check.validate(STORE, "nowhere.py")
# Within the caller's transaction, a module's visitor is read; the store and a path open their own
with STORE.transaction():
    module = STORE.singleton(Model.WORKSPACE).files({"path": path})
    assert Check.check_file(STORE, module, (3, 11)) == ["body[0]: TypeAlias needs Python 3.12"]
    assert Check.validate_file(module) == []
    with raises(ValueError, match="transactions do not nest"):
        Check.check(STORE, path, (3, 11))
# A store that holds the installs alone has no versions; one with the language's alone has no terms' versions
bare = Python3.store()
bare.transaction(Parse.parse("pass\n", "a.py"))
with raises(LookupError, match="run Standards.prepare"):
    Check.check(bare, "a.py", (3, 12))
bare.transaction(Standards.install)
bare.transaction(Standards.define)
with raises(LookupError, match="Expressions.Python.Python3 covers no version"):
    Check.check(bare, "a.py", (3, 12))
assert Check.check(Standards.prepare(bare), "a.py", (3, 0)) == []
print("ok")

## P3CHK-04 · The conformance source: none of its constructs lacking in Python 3.14, nor 3.15's added to it; in older versions, each construct where it arrived

In [ ]:
text = (SOURCES / "python314.py").read_text(encoding="utf-8")
path = parsed(text)
assert Check.check(STORE, path, (3, 14)) == Check.check(STORE, path, (3, 15)) == []
assert Check.check(STORE, path, (3, 13)) == ["body[19].body[1].value: templatestr needs Python 3.14"]
problems = Check.check(STORE, path, (3, 9))
assert all(p.endswith(("3.10", "3.11", "3.12", "3.13", "3.14")) for p in problems), problems
assert problems[:3] == ["body[9]: TypeAlias needs Python 3.12", "body[9].type_params[0]: typevar needs Python 3.12",
                        "body[9].type_params[1]: typevar needs Python 3.12"]
oldest = Check.check(STORE, path, (3, 0))
features = {p.split(": ", 1)[1].rsplit(" needs ", 1)[0] for p in oldest}
assert features >= {"TypeAlias", "TryStar", "Match", "MatchCase", "AnnAssign", "AsyncFunctionDef", "AsyncWith",
                    "AsyncFor", "await", "yieldfrom", "joinedstr", "namedexpr", "templatestr", "matchor", "typevar",
                    "paramspec", "typevartuple", "ClassDef.type_params", "binop.op '@'", "AugAssign.op '@'",
                    "arguments.posonlyargs", "comprehension.is_async", "formattedvalue.debug",
                    "subscript with an unpacked index", "typevar.default_value",
                    "ClassDef with a decorator holding any term"}, features
assert len(oldest) == len(set(oldest)) and set(problems) <= set(oldest)
# Python 3.15's constructs added to it: only they lack in 3.14
later = parsed(text + "\nlazy import json\nlazy from os import path\nflat = [*row for row in rows]\n"
               "merged = {**m for m in maps}\n")
assert Check.check(STORE, later, (3, 15)) == []
assert Check.check(STORE, later, (3, 14)) == [
    "body[25].is_lazy: Import.is_lazy needs Python 3.15", "body[26].is_lazy: ImportFrom.is_lazy needs Python 3.15",
    "body[27].value: listcomp of unpacked elements needs Python 3.15",
    "body[28].value: dictcomp of unpacked mappings needs Python 3.15"]
assert Check.validate(STORE, path) == [] and Check.validate(STORE, later) == []
print(len(oldest), "constructs lacking in Python 3.0")

## P3CHK-05 · Each kind's own rules: what it needs, what its blocks and lists must hold, what goes together, and the names it declares; where a statement or a clause stands, the schemas enforce on write

A block (`body`, `orelse`, `finalbody`) is a `Programs.Python3.Body`, whose union has no clause branch; a `try`'s
`handlers` take only an `except` clause or an error, a `match`'s `cases` only a `case` or an error; a kind has only its
own properties. So a misplaced statement or clause is never stored: writing one fails, an `AttributeError` for a branch
the union does not have, a `ReadOnlyError` (a `TypeError`) for a property the kind does not have, and `validate` has no
rule for it.

In [ ]:
problems = [
    (lambda u: u.assign(lambda s: s.value(P.name("a"))), "body[0]: an Assign needs a target"),
    (lambda u: u.delete(lambda s: s), "body[0]: a Delete needs a target"),
    (lambda u: getattr(u, "import")(lambda s: s), "body[0]: an Import needs a name"),
    (lambda u: u.importfrom(lambda s: s.module("m")), "body[0]: an ImportFrom needs a name"),
    (lambda u: u.importfrom(lambda s: s.names([lambda a: a.name("*")])),
     "body[0]: an ImportFrom needs a module or a level"),
    (lambda u: getattr(u, "global")(lambda s: s), "body[0]: a Global needs a name"),
    (lambda u: getattr(u, "nonlocal")(lambda s: s), "body[0]: a Nonlocal needs a name"),
    (lambda u: getattr(u, "raise")(lambda s: s.cause(P.name("c"))), "body[0]: a Raise with a cause needs an exc"),
    (lambda u: u.expr(lambda s: s), "body[0]: an Expr needs a value"),
    (lambda u: getattr(u, "assert")(lambda s: s), "body[0]: an Assert needs a test"),
    (lambda u: getattr(u, "if")(lambda s: s.body([PASS])), "body[0]: an If needs a test"),
    (lambda u: getattr(u, "if")(lambda s: s.test(P.name("a")).body([COMMENT])),
     "body[0]: an If needs a statement in its body"),
    (lambda u: getattr(u, "if")(lambda s: s.test(P.name("a")).body([PASS]).orelse([COMMENT])),
     "body[0]: an If needs a statement in its orelse"),
    (lambda u: getattr(u, "while")(lambda s: s.test(P.name("a"))), "body[0]: a While needs a statement in its body"),
    (lambda u: getattr(u, "for")(lambda s: s.target(P.name("a")).iter(P.name("b"))),
     "body[0]: a For needs a statement in its body"),
    (lambda u: u.asyncfor(lambda s: s.target(P.name("a")).iter(P.name("b"))),
     "body[0]: an AsyncFor needs a statement in its body"),
    (lambda u: getattr(u, "try")(lambda s: s.body([PASS])), "body[0]: a Try needs a handler or a finalbody"),
    (lambda u: getattr(u, "try")(lambda s: s.body([PASS]).orelse([PASS]).finalbody([PASS])),
     "body[0]: a Try with an orelse needs a handler"),
    (lambda u: u.trystar(lambda s: s.body([PASS])), "body[0]: a TryStar needs a handler or a finalbody"),
    (lambda u: getattr(u, "try")(lambda s: s.body([PASS]).handlers([lambda h: h.excepthandler(lambda c: c)])),
     "body[0].handlers[0]: an ExceptHandler needs a statement in its body"),
    (lambda u: getattr(u, "with")(lambda s: s.body([PASS])), "body[0]: a With needs an item"),
    (lambda u: u.asyncwith(lambda s: s.items([lambda w: w.context_expr(P.name("a"))])),
     "body[0]: an AsyncWith needs a statement in its body"),
    (lambda u: u.match(lambda s: s.subject(P.name("a"))), "body[0]: a Match needs a case"),
    (lambda u: u.match(lambda s: s.subject(P.name("a")).cases([lambda c: c.matchcase(lambda m: m.body([PASS]))])),
     "body[0].cases[0]: a MatchCase needs a pattern"),
    (lambda u: u.functiondef(lambda s: s.args(lambda a: a).body([PASS])), "body[0]: a FunctionDef needs a name"),
    (lambda u: u.asyncfunctiondef(lambda s: s.name("f").args(lambda a: a)),
     "body[0]: an AsyncFunctionDef needs a statement in its body"),
    (lambda u: u.classdef(lambda s: s.name("C")), "body[0]: a ClassDef needs a statement in its body"),
    (lambda u: u.augassign(lambda s: s.target(P.name("a")).op("@@").value(P.name("b"))),
     "body[0]: AugAssign.op cannot be '@@'"),
    (lambda u: u.typealias(lambda s: s.name(P.name("X"))), "body[0]: a TypeAlias needs a value"),
]
for statement, problem in problems:
    found = validate(statement)
    assert found == [problem], (problem, found)
# What needs several, each
assert validate(lambda u: u.augassign(lambda s: s)) == [
    "body[0]: an AugAssign needs a target", "body[0]: an AugAssign needs an op", "body[0]: an AugAssign needs a value"]
assert validate(lambda u: getattr(u, "for")(lambda s: s.body([PASS]))) == [
    "body[0]: a For needs a target", "body[0]: a For needs an iter"]
# Names a statement declares: identifiers, not keywords; dotted names; `*` only in a from-import
assert validate(lambda u: u.functiondef(lambda s: s.name("class").args(lambda a: a).body([PASS]))) == [
    "body[0].name: 'class' is a keyword"]
assert validate(lambda u: u.classdef(lambda s: s.name("1x").body([PASS]))) == [
    "body[0].name: '1x' is not an identifier"]
assert validate(lambda u: getattr(u, "global")(lambda s: s.names(["a", "None", "b-c"]))) == [
    "body[0].names[1]: 'None' is a keyword", "body[0].names[2]: 'b-c' is not an identifier"]
assert validate(lambda u: getattr(u, "try")(lambda s: s.body([PASS]).handlers([
    lambda h: h.excepthandler(lambda c: c.name("for").body([PASS]))]))) == [
    "body[0].handlers[0].name: 'for' is a keyword"]
assert validate(lambda u: getattr(u, "import")(lambda s: s.names([
    lambda a: a.name("a.b").asname("if"), lambda a: a.name("a..b"), lambda a: a.name("*"),
    lambda a: a.asname("x")]))) == [
    "body[0].names[0]: 'if' is a keyword", "body[0].names[1]: '' is not an identifier",
    "body[0].names[2]: '*' is not an identifier", "body[0].names[3]: an alias needs a name"]
assert validate(lambda u: u.importfrom(lambda s: s.module("a.class").names([lambda a: a.name("*")]))) == [
    "body[0].module: 'class' is a keyword"]
for spelling in ("x_1", "Größe", "_", "match", "case", "type", "lazy", "print"):  # soft keywords are identifiers
    assert validate(lambda u: getattr(u, "nonlocal")(lambda s: s.names([spelling]))) == [], spelling
# Where a statement or a clause stands: the schemas refuse it on write, a clause in a block, a statement among a try's
# handlers, a block or a clause list a kind does not have
with raises(AttributeError, match="excepthandler"):  # a module's body has no clause branch
    written(lambda u: u.excepthandler(lambda c: c.body([PASS])))
with raises(AttributeError, match="pass"):  # a try's handlers take only an except clause or an error
    written(lambda u: getattr(u, "try")(lambda s: s.body([PASS]).handlers([PASS])))
with raises(AttributeError, match="matchcase"):  # nor a case
    written(lambda u: getattr(u, "try")(lambda s: s.body([PASS]).handlers([
        lambda h: h.matchcase(lambda c: c.pattern(lambda p: p.matchas(lambda r: r)).body([PASS]))])))
with raises(AttributeError, match="excepthandler"):  # a match's cases take only a case or an error
    written(lambda u: u.match(lambda s: s.subject(P.name("a")).cases([lambda c: c.excepthandler(lambda h: h)])))
with raises(TypeError, match="'cases' is not a member"):  # an if has no cases
    written(lambda u: getattr(u, "if")(lambda s: s.test(P.name("a")).body([PASS]).cases([
        lambda c: c.matchcase(lambda m: m.pattern(lambda p: p.matchas(lambda r: r)).body([PASS]))])))
with raises(TypeError, match="'body' is not a member"):  # a match has no body
    written(lambda u: u.match(lambda s: s.subject(P.name("a")).cases([
        lambda c: c.matchcase(lambda m: m.pattern(lambda p: p.matchas(lambda r: r)).body([PASS]))]).body([PASS])))
with raises(TypeError, match="'body' is not a member"):  # a pass has no block
    written(lambda u: getattr(u, "pass")(lambda s: s.body([PASS])))
# What is fine
assert validate(
    lambda u: getattr(u, "try")(lambda s: s.body([PASS]).handlers([lambda h: h.excepthandler(lambda c: c.body([PASS]))])
                                .orelse([PASS])),
    lambda u: getattr(u, "try")(lambda s: s.body([PASS]).finalbody([COMMENT, PASS])),
    lambda u: u.importfrom(lambda s: s.level(1).names([lambda a: a.name("*")])),
    lambda u: getattr(u, "raise")(lambda s: s), lambda u: getattr(u, "return")(lambda s: s), COMMENT,
) == []
print(len(problems), "problems")

## P3CHK-06 · Terms as their dialects validate them, located by the property that holds them; names they refer to taken as bound

In [ ]:
assert validate(lambda u: u.expr(lambda s: s.value(P.set_()))) == ["body[0].value: a set needs an element"]
assert validate(lambda u: u.expr(lambda s: s.value(P.boolop("and", P.name("a"))))) == [
    "body[0].value: a boolop needs two values"]
lambda_ = P.lambda_(P.name("a"), [P.arg("a", annotation=P.name("int"))])
assert validate(lambda u: u.expr(lambda s: s.value(lambda_))) == [
    "body[0].value: a lambda's parameters have no annotations"]
assert validate(lambda u: u.match(lambda s: s.subject(P.name("a")).cases([lambda c: c.matchcase(lambda m: m
    .pattern(lambda p: p.matchor(lambda r: r.patterns([lambda q: q.matchas(lambda a: a)]))).body([PASS]))]))) == [
    "body[0].cases[0].pattern: a MatchOr needs two patterns"]
assert validate(lambda u: u.assign(lambda s: s.targets([P.name("a"), P.set_()]).value(P.name("undefined")))) == [
    "body[0].targets[1]: a set needs an element"]
assert validate(lambda u: u.functiondef(lambda s: s.name("f").args(lambda a: a.args([
    P.arg("a", default_value=P.set_())])).body([PASS]))) == ["body[0].args: a set needs an element"]
# A whole module, parsed: none
assert Check.validate(STORE, parsed("def f(a, *, b=1):\n    return [x for x in a if (y := x)]\n")) == []
print("ok")

## P3CHK-07 · Printing writes what the store holds; checking it against a version is `check`'s, and validating it `validate`'s (ported from PYPRT-06)

In [ ]:
path = parsed("t'x'\n")
assert Print.print_module(STORE, path) == "t'x'\n"  # printed, whatever the version
assert Check.check(STORE, path, (3, 12)) == ["body[0].value: templatestr needs Python 3.14"]
path = parsed("lazy import a\n")
assert Print.print_module(STORE, path) == "lazy import a\n"
assert Check.check(STORE, path, (3, 14)) == ["body[0].is_lazy: Import.is_lazy needs Python 3.15"]
path = parsed("(x := a)\nreturn (x := a)\nfor x in (x := a):\n    pass\n")
assert Check.check(STORE, path, (3, 12)) == [] and Check.validate(STORE, path) == []
assert Print.print_module(STORE, path) == "(x := a)\nreturn (x := a)\nfor x in (x := a):\n    pass\n"
assert validate(lambda u: getattr(u, "if")(lambda s: s.body([PASS]))) == ["body[0]: an If needs a test"]
assert validate(lambda u: u.functiondef(lambda s: s.name("class").args(lambda a: a).body([PASS]))) == [
    "body[0].name: 'class' is a keyword"]
print("ok")

## P3CHK-08 · Invalid syntax, which a parse keeps: `validate` reports each `Error` and each `error` and `missing` term, located; `check` skips them and checks what was recovered within an `Error`; an `Error` stands where a statement or a clause may, and holds statements, the schemas refusing a clause or a block it does not have

In [ ]:
# Each `Error` and each `error` and `missing` term, located; nothing else of a parsed file of good and bad statements
path = parsed("import os\n"
              "x = = 1\n"
              "def f(a):\n"
              "    y = (a + )\n"
              "    print 'old'\n"
              "    for in a:\n"
              "        pass\n"
              "class C(:\n"
              "    z = 1\n"
              "return return\n")
assert Check.validate(STORE, path) == [
    "body[1]: invalid syntax 'x = = 1'",
    "body[2].body[0].value: invalid syntax '(a + )'",
    "body[2].body[1]: invalid syntax \"print 'old'\"",
    "body[2].body[2].target: missing identifier",
    "body[3]: invalid syntax 'class C(:\\n    z = 1'",
    "body[5]: invalid syntax ''",  # a newline found missing
]
# The statements recovered within an `Error` are checked as any are, within it
assert Check.validate(STORE, parsed("class C(:\n    x = (1 + )\n")) == [
    "body[0]: invalid syntax 'class C(:\\n    x = (1 + )'", "body[0].body[0].value: invalid syntax '(1 + )'"]
# `check` skips invalid syntax, which no standard lists, and checks what was recovered within it
assert Check.check(STORE, path, (3, 0)) == []
assert Check.check(STORE, parsed("class C(:\n    type X = int\n"), (3, 11)) == ["body[0].body[0]: TypeAlias needs Python 3.12"]
# An `Error` stands where a statement or a clause may, and holds statements in its body, not clauses: the schemas
# refuse a clause there, or a block it does not have, on write
assert validate(lambda u: getattr(u, "try")(lambda s: s
    .body([lambda b: b.error(lambda e: e.text("x = = 1"))])
    .handlers([lambda h: h.error(lambda e: e.text("except E +:"))])
    .finalbody([lambda b: b.error(lambda e: e.text("pass").body([PASS]))]))) == [
    "body[0].body[0]: invalid syntax 'x = = 1'", "body[0].handlers[0]: invalid syntax 'except E +:'",
    "body[0].finalbody[0]: invalid syntax 'pass'"]
assert validate(lambda u: u.match(lambda s: s.subject(P.name("a")).cases([lambda c: c.error(lambda e: e.text("case")
    .body([PASS]))]))) == ["body[0].cases[0]: invalid syntax 'case'"]
with raises(AttributeError, match="excepthandler"):
    written(lambda u: u.error(lambda e: e.text("except:").body([lambda b: b.excepthandler(lambda c: c.body([PASS]))])))
with raises(TypeError, match="'orelse' is not a member"):
    written(lambda u: u.error(lambda e: e.text("").orelse([PASS])))
print("ok")

## P3CHK-09 · Grammar clauses as data: the language owns its statements' by name, each allowed or forbidden by each version, the terms' its dialects'; every former table entry by data, PEP 614 both ways, an unrepresented version guessed by its neighbour's flag, and a grammar clause no version names allowed

What `Check`'s code tables (`STATEMENT_FEATURES` and `FEATURES`) held is data: the statements' grammar clauses
`Programs.Python3`'s, written by `Standards.define` through mbse-expressions' writer, the terms' mbse-expressions'
dialects'. PEP 614 removed a restriction, so it is two grammar clauses of each definition kind, with opposite senses of
version progression, which `Check` reads as alternatives: a dotted name, optionally called, takes both and is fine in
every version; any other term takes the second alone.

In [ ]:
DEFINITIONS = ("FunctionDef", "AsyncFunctionDef", "ClassDef")
DOTTED, ANY = "with a decorator holding a dotted name, optionally called", "with a decorator holding any term"
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    every = list(ExpressionStandards.standards(language))

    def allowed(dialect, name):
        """The versions `dialect` covers that allow its grammar clause `name`."""
        singleton = STORE.singleton(dialect)
        clause = ExpressionStandards.grammar_clauses(singleton)[name]
        return [v for v, s in ExpressionStandards.standards(singleton).items() if ExpressionStandards.allows(s, clause)]

    clauses = ExpressionStandards.grammar_clauses(language)
    assert list(clauses) == [clause.name for clause in Standards.GRAMMAR_CLAUSES] and len(clauses) == 14
    assert not hasattr(Check, "FEATURES") and not hasattr(Check, "STATEMENT_FEATURES")  # data, not code
    # `STATEMENT_FEATURES`' entries, the language's
    said = ExpressionStandards.grammar_clause(clauses["Import.is_lazy"])
    assert said["kind"].name == "Programs.Python3.Import" and said["property"] == "is_lazy"
    assert said["values"] == (True,)
    assert allowed(LANGUAGE, "Import.is_lazy") == allowed(LANGUAGE, "ImportFrom.is_lazy") == every[15:]
    assert allowed(LANGUAGE, "AugAssign.op '@'") == every[5:]
    assert ExpressionStandards.grammar_clause(clauses["ClassDef.type_params"])["present"] is True
    assert all(allowed(LANGUAGE, f"{kind}.type_params") == every[12:] for kind in DEFINITIONS)
    # PEP 614 both ways: a dotted name, optionally called, 3.0 to 3.8's, forbidden from 3.9; any term forbidden before
    # 3.9, allowed from it
    for kind in DEFINITIONS:
        assert allowed(LANGUAGE, f"{kind} {DOTTED}") == every[:9] and allowed(LANGUAGE, f"{kind} {ANY}") == every[9:]
    dotted = ExpressionStandards.grammar_clause(clauses[f"FunctionDef {DOTTED}"])
    assert [kind.name.rsplit(".", 1)[-1] for kind in dotted["kinds"]] == ["Variable", "Attribute", "Call"]
    assert [ExpressionStandards.grammar_clause_name(n) for n in dotted["nested"]] == [
        "attribute of a dotted name", "call of a dotted name"]
    assert len(ExpressionStandards.grammar_clause(clauses[f"ClassDef {ANY}"])["kinds"]) == len(Standards.ANY) == 30
    # Nested, no version names them: allowed by each (the default)
    for name in ("attribute of a dotted name", "call of a dotted name"):
        assert allowed(LANGUAGE, name) == every and not Objects.entries(clauses[name], "allowed")
    # `FEATURES`' entries, the terms', mbse-expressions' dialects'
    first = {"binop.op '@'": 5, "comprehension.is_async": 6, "formattedvalue.debug": 8, "arguments.posonlyargs": 8,
             "subscript with an unpacked index": 11, "listcomp of unpacked elements": 15,
             "setcomp of unpacked elements": 15, "generatorexp of unpacked elements": 15,
             "dictcomp of unpacked mappings": 15}
    assert all(allowed("Expressions.Python.Python3", name) == every[minor:] for name, minor in first.items())
    assert all(allowed("Expressions.Python.Types", f"{tag}.default_value") == every[13:]
               for tag in ("typevar", "paramspec", "typevartuple"))
    assert allowed("Expressions.Python.Python3", "tuple with an unpacked element") == every  # nested: the default
# Through `check`, PEP 614 both ways, located at each decorator that takes the grammar clause
for kind, text in (("FunctionDef", "def f():\n    pass\n"), ("AsyncFunctionDef", "async def f():\n    pass\n"),
                   ("ClassDef", "class C:\n    pass\n")):
    path = parsed("@d\n@d.e\n@d.e(1)\n@(d)\n@d[0]\n@d(1)(2)\n@d(1).e\n@d.e.f\n" + text)
    assert Check.check(STORE, path, (3, 9)) == Check.check(STORE, path, (3, 15)) == []
    assert Check.check(STORE, path, (3, 8)) == [
        f"body[0].decorator_list[{i}]: {kind} {ANY} needs Python 3.9" for i in (3, 4, 5, 6)]
# A grammar clause no version names is allowed: a tuple with an unpacked element, which Python3 nests in a subscript's
assert check("(a, *b)\n", (3, 5)) == [] and check("a[b, *c]\n", (3, 10)) == [
    "body[0].value: subscript with an unpacked index needs Python 3.11"]
# An unrepresented version, Python 3.16, which a store adds with Python 3.15's kinds and no grammar clause's entries:
# guessed by Python 3.15's entries, flagged `after`, the language's and the dialects' alike
future = Standards.prepare(Python3.store())
with future.transaction():
    for dialect in Standards.DIALECTS:
        singleton = future.singleton(dialect)
        kinds = ExpressionStandards.kinds(ExpressionStandards.find(singleton, "Python 3.15"))
        standard = future.Expressions.Framework.Standard(lambda s: s.bodies(["org.python"]).revision(
            lambda r: r.version("3.16")).kinds(kinds))
        holder = reduce(getattr, f"{dialect}.{'Language' if dialect == LANGUAGE else 'Dialect'}".split("."), future)
        holder(singleton, lambda b: b.standards({"Python 3.16": lambda e: e.standard(standard)}))
future.transaction(Parse.parse("lazy import m\nx = a @ b\n@d[0]\n@d\ndef f[T = int]():\n    pass\n", "future.py"))
assert Check.check(future, "future.py", (3, 16)) == Check.check(future, "future.py", "Python 3.16") == []
with future.transaction():
    language = future.singleton(LANGUAGE)
    clauses, newest = ExpressionStandards.grammar_clauses(language), ExpressionStandards.find(language, "Python 3.16")
    assert not Objects.entries(newest, "allows") and not Objects.entries(newest, "forbids")  # unrepresented
    assert ExpressionStandards.allows(newest, clauses["Import.is_lazy"])  # 3.15's `Allows`, flagged `after`
    assert not ExpressionStandards.allows(newest, clauses[f"FunctionDef {DOTTED}"])  # 3.15's `Forbids`, flagged `after`
    assert ExpressionStandards.allows(newest, clauses[f"FunctionDef {ANY}"])  # its alternative: `@d` fine
print("ok")